# Protein Language Models Tutorial

This notebook demonstrates how to work with Protein Language Models (pLMs) using Hugging Face Transformers. We'll cover:
- Loading pre-trained pLMs
- Fine-tuning on protein-related tasks
- Evaluating model performance
- Experimenting with different models and tasks

## 1. Install and Import Libraries

First, let's install the necessary libraries and import them.

In [4]:
# Install required packages (run this in a terminal if not already installed)
# !pip install transformers torch datasets evaluate scikit-learn ipywidgets

# Import libraries
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Libraries imported successfully!
PyTorch version: 2.11.0+cpu
CUDA available: False


## 2. Load a Pre-trained Protein Language Model

Protein Language Models are transformer models trained on large protein sequence datasets. We'll use ESM-2, a smaller variant that's good for learning and experimentation.

Popular pLMs include:
- ESM-1b/2 (Meta AI)
- ProtBERT/BERT (Rostlab)
- ProtT5 (Technical University of Munich)

In [8]:
# Model selection widget
model_options = {
    "ESM-2 (8M)": {
        "name": "facebook/esm2_t6_8M_UR50D",
        "year": "2022",
        "publication": "Lin et al., bioRxiv 2022",
        "description": "Evolutionary Scale Modeling 2 - 8M parameters. Trained on UniRef50. Good for small-scale experiments."
    },
    "ESM-2 (35M)": {
        "name": "facebook/esm2_t12_35M_UR50D",
        "year": "2022",
        "publication": "Lin et al., bioRxiv 2022",
        "description": "Evolutionary Scale Modeling 2 - 35M parameters. Better performance than 8M variant."
    },
    "ESM-2 (150M)": {
        "name": "facebook/esm2_t30_150M_UR50D",
        "year": "2022",
        "publication": "Lin et al., bioRxiv 2022",
        "description": "Evolutionary Scale Modeling 2 - 150M parameters. Good balance of performance and speed."
    },
    "ProtBERT": {
        "name": "Rostlab/prot_bert",
        "year": "2019",
        "publication": "Elnaggar et al., IEEE Transactions on Computational Biology and Bioinformatics 2019",
        "description": "BERT model adapted for proteins. Based on BERT-base architecture."
    },
    "ProtT5-XL": {
        "name": "Rostlab/prot_t5_xl_uniref50",
        "year": "2021",
        "publication": "Elnaggar et al., Nature Communications 2021",
        "description": "T5 model for proteins - XL variant. Strong performance on various protein tasks."
    }
}

# Create dropdown widget
model_dropdown = widgets.Dropdown(
    options=list(model_options.keys()),
    value="ESM-2 (8M)",  # Default
    description='Model:',
    style={'description_width': 'initial'}
)

# Output widget for model info
model_info_output = widgets.Output()

def on_model_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_model = change['new']
        model_data = model_options[selected_model]

        with model_info_output:
            clear_output(wait=True)
            print(f"📊 Model: {selected_model}")
            print(f"🔗 Hugging Face ID: {model_data['name']}")
            print(f"📅 Year: {model_data['year']}")
            print(f"📝 Publication: {model_data['publication']}")
            print(f"📖 Description: {model_data['description']}")

            # Set the global MODEL_NAME
            global MODEL_NAME
            MODEL_NAME = model_data['name']
            print(f"\n✅ MODEL_NAME set to: {MODEL_NAME}")

# Connect the callback
model_dropdown.observe(on_model_change)

# Display widgets
display(model_dropdown)
display(model_info_output)

# Initialize with default model info and set MODEL_NAME
selected_model = model_dropdown.value
model_data = model_options[selected_model]
print(f"📊 Model: {selected_model}")
print(f"🔗 Hugging Face ID: {model_data['name']}")
print(f"📅 Year: {model_data['year']}")
print(f"📝 Publication: {model_data['publication']}")
print(f"📖 Description: {model_data['description']}")

MODEL_NAME = model_data['name']
print(f"\n✅ MODEL_NAME set to: {MODEL_NAME}")

# Update display when model changes
def on_model_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_model = change['new']
        model_data = model_options[selected_model]

        with model_info_output:
            clear_output(wait=True)
            print(f"📊 Model: {selected_model}")
            print(f"🔗 Hugging Face ID: {model_data['name']}")
            print(f"📅 Year: {model_data['year']}")
            print(f"📝 Publication: {model_data['publication']}")
            print(f"📖 Description: {model_data['description']}")

            # Update the global MODEL_NAME
            global MODEL_NAME
            MODEL_NAME = model_data['name']
            print(f"\n✅ MODEL_NAME updated to: {MODEL_NAME}")

# Connect the callback
model_dropdown.observe(on_model_change)

Dropdown(description='Model:', options=('ESM-2 (8M)', 'ESM-2 (35M)', 'ESM-2 (150M)', 'ProtBERT', 'ProtT5-XL'),…

Output()

📊 Model: ESM-2 (8M)
🔗 Hugging Face ID: facebook/esm2_t6_8M_UR50D
📅 Year: 2022
📝 Publication: Lin et al., bioRxiv 2022
📖 Description: Evolutionary Scale Modeling 2 - 8M parameters. Trained on UniRef50. Good for small-scale experiments.

✅ MODEL_NAME set to: facebook/esm2_t6_8M_UR50D


In [6]:
# Load the selected model
print(f"Loading model: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model for sequence classification (we'll modify num_labels based on task)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # Binary classification (soluble/insoluble)
    problem_type="single_label_classification"
)

print(f"Model loaded successfully!")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model architecture: {model.__class__.__name__}")

Loading model: facebook/esm2_t6_8M_UR50D


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 13843.48it/s]
[transformers] EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Number of parameters: 7,512,443
Model architecture: EsmForSequenceClassification


## 3. Prepare Protein Sequence Data

We'll create a synthetic protein dataset for demonstration purposes, simulating protein sequences labeled as soluble or insoluble. This allows us to demonstrate the pLM fine-tuning process without requiring access to external datasets.

**Dataset Details**: 
- **Synthetic Data**: Randomly generated protein sequences with binary solubility labels
- **Training set**: 500 samples
- **Test set**: 100 samples
- **Task**: Binary classification (soluble vs insoluble)

**Note**: You can try loading real datasets by setting `loading_external_dataset = True` in the code cell below. In a real scenario, you would use actual experimental data such as:
- DeepSol dataset (when available on Hugging Face)
- Secondary structure prediction datasets
- Protein function annotation datasets
- Stability prediction datasets

For real protein data, you can explore:
- UniProt database
- Protein Data Bank (PDB)
- Various datasets on Hugging Face Hub

In [15]:
# Set this flag to True to try loading external datasets, False to use synthetic data
loading_external_dataset = False

if loading_external_dataset:
    # Try to load the DeepSol dataset from Hugging Face
    print("Attempting to load DeepSol dataset from Hugging Face...")
    try:
        dataset = load_dataset("proteinea/solubility")
        dataset = dataset.rename_columns({"sequences": "sequence", "labels": "label"})
        dataset = dataset.map(lambda b: {"label": [int(x) for x in b["label"]]}, batched=True)
        dataset = dataset.shuffle(seed=42)
        print("✓ Successfully loaded DeepSol dataset!")
        
        # For demo purposes, use a small subset
        train_dataset = dataset['train'].select(range(500))  # 500 training samples
        test_dataset = dataset['test'].select(range(100))   # 100 test samples
        
    except Exception as e:
        print(f"✗ Failed to load DeepSol dataset: {e}")
        print("Falling back to synthetic data...")
        loading_external_dataset = False

if not loading_external_dataset:
    # Create a synthetic protein dataset for demonstration
    print("Creating synthetic protein dataset for demonstration...")

    # Generate synthetic protein sequences and labels
    import random

    def generate_protein_sequence(length=100):
        """Generate a random protein sequence."""
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        return ''.join(random.choice(amino_acids) for _ in range(length))

    # Create synthetic data
    n_train = 500
    n_test = 100

    train_sequences = [generate_protein_sequence(random.randint(50, 150)) for _ in range(n_train)]
    train_labels = [random.choice([0, 1]) for _ in range(n_train)]  # 0: insoluble, 1: soluble

    test_sequences = [generate_protein_sequence(random.randint(50, 150)) for _ in range(n_test)]
    test_labels = [random.choice([0, 1]) for _ in range(n_test)]

    # Create datasets using datasets library
    from datasets import Dataset, DatasetDict

    train_dataset = Dataset.from_dict({
        'sequence': train_sequences,
        'label': train_labels
    })

    test_dataset = Dataset.from_dict({
        'sequence': test_sequences,
        'label': test_labels
    })

    dataset = DatasetDict({
        'train': train_dataset,
        'test': test_dataset
    })

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Check the structure
print("\nDataset structure:")
print(train_dataset[0])

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["sequence"],
        truncation=True,
        padding=False,  # We'll use dynamic padding
        max_length=512
    )

# Tokenize datasets
print("\nTokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["sequence"])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["sequence"])

print("Data preparation complete!")

Creating synthetic protein dataset for demonstration...
Training samples: 500
Test samples: 100

Dataset structure:
{'sequence': 'EAKIIFEVDWQCADHITYAVHVQIRWKAGQMKFHMEDPENNYKCRVEPDVLYNWHDCILDIEPKRNGNNHKDYGVIGRPKVIMCICMPKDHWMHSPRFKFIVVKWQWPNIFTSDCEFGQYDPPYRTKVAEV', 'label': 0}

Tokenizing datasets...


Map: 100%|██████████| 100/100 [00:00<00:00, 3226.74 examples/s]

Data preparation complete!


## 4. Fine-tune the Model on a Task

Fine-tuning adapts the pre-trained pLM to our specific task (solubility prediction). This process updates the model's weights using our labeled data.

The process involves:
1. Setting up training arguments (learning rate, batch size, epochs)
2. Creating a Trainer object
3. Training the model
4. Saving the fine-tuned model

In [13]:
# Define metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,  # Small batch size for demo
    per_device_eval_batch_size=4,
    num_train_epochs=2,  # Few epochs for demo
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
print("Starting fine-tuning...")
trainer.train()

# Save the model
trainer.save_model("./fine_tuned_model")
print("Model saved to ./fine_tuned_model")

Starting fine-tuning...


c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.694923,0.690549,0.560000,0.578947,0.932203,0.714286
2,0.686506,0.694715,0.490000,0.576923,0.508475,0.540541


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 56.98it/s]
c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 79.82it/s]

Model saved to ./fine_tuned_model


## 5. Evaluate Model Performance

After fine-tuning, we evaluate the model's performance on the test set using various metrics. This helps us understand how well the model generalizes to unseen data.

In [14]:
# Evaluate the model
print("Evaluating model on test set...")
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
for metric, value in eval_results.items():
    print(f"{metric}: {value:.4f}")

# You can also make predictions on individual examples
print("\nExample predictions:")
predictions = trainer.predict(tokenized_test)
pred_labels = np.argmax(predictions.predictions, axis=1)

# Show a few examples
for i in range(5):
    true_label = test_dataset[i]['label']
    pred_label = pred_labels[i]
    sequence = test_dataset[i]['sequence'][:50] + "..."
    status = "✓" if true_label == pred_label else "✗"
    print(f"{status} True: {true_label}, Pred: {pred_label} | {sequence}")

Evaluating model on test set...


c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.686506,0.690549,2,0.560000,0.578947,0.932203,0.714286



Evaluation Results:
eval_loss: 0.6905
eval_accuracy: 0.5600
eval_precision: 0.5789
eval_recall: 0.9322
eval_f1: 0.7143

Example predictions:


c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✓ True: 1, Pred: 1 | TPNPKEAIFMRRDPKMVPHTIQDVTGIWYSEDKWYRIFIGCIWGAFNYWK...
✓ True: 1, Pred: 1 | YVLGTETVYGNMKGDGHSMIRNTSGQYKQFPAPWWIHLQDKYTKFSQLWR...
✗ True: 0, Pred: 1 | PWMSEQGIGMMEPMGRIWGVDGRAKMEHWMNSRSQHQQLANVGWVCLQLD...
✗ True: 0, Pred: 1 | YQIKAARPFWCAYFKWFQLSWDRDMTLASCPKHRQLVFQKCSYHAYMRYW...
✓ True: 1, Pred: 1 | DWHYKHNWNNRAWMQTKAYGMIINFQFGYQWGRAKQDRGWAYDSTQHNYH...


## 6. Test with Different Models and Tasks

To experiment with other models and tasks:

### Different Models:
- `"Rostlab/prot_bert"` - ProtBERT model
- `"facebook/esm2_t12_35M_UR50D"` - Larger ESM-2 model
- `"facebook/esm1v_t33_650M_UR90S_1"` - ESM-1v for variant effect prediction

### Different Tasks:
- **Secondary Structure Prediction**: Use token classification instead of sequence classification
- **Protein Function Prediction**: Multi-label classification
- **Stability Prediction**: Regression task

### Example: Try a Different Model
Uncomment and modify the code below to try different models.

In [ ]:
# Example: Try a different model
# Uncomment the lines below to try ProtBERT instead

# MODEL_NAME = "Rostlab/prot_bert"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=2,
#     problem_type="single_label_classification"
# )
# print(f"Switched to model: {MODEL_NAME}")

# For regression tasks (e.g., stability prediction), change to:
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=1,  # Single output for regression
#     problem_type="regression"
# )

# For token classification (e.g., secondary structure):
# from transformers import AutoModelForTokenClassification
# model = AutoModelForTokenClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=3  # H, E, C for helix, sheet, coil
# )

print("Experiment with different models by changing MODEL_NAME and adjusting the model type!")
print("\nNext steps:")
print("1. Try different pre-trained models using the dropdown widget")
print("2. Set loading_external_dataset = True to try loading real protein datasets")
print("3. Experiment with different datasets/tasks")
print("4. Adjust hyperparameters (learning rate, batch size, epochs)")
print("5. Use the model for inference on new sequences")

# Expanding to Other Task Types

Sections 1–6 covered **solubility prediction** (binary sequence classification). The same
pre-trained pLMs can be adapted to fundamentally different problem shapes just by swapping
the task head and loss. The next three sections demonstrate each one on synthetic data, and
section 10 benchmarks **all 4 models on all 4 tasks** in a single comparison.

| Section | Task | Problem type | Head / loss |
|---------|------|--------------|-------------|
| 7 | Secondary structure | Token (per-residue) classification | `AutoModelForTokenClassification` |
| 8 | Protein function | Multi-label classification | sigmoid + BCE |
| 9 | Stability | Regression | single linear output + MSE |

Each section reuses the synthetic sequences from section 3 and the model selected in
section 2 (via `MODEL_NAME` / `tokenizer`), so make sure those cells have been run first.

## 7. Secondary Structure Prediction (Token Classification)

Secondary structure (α-helix, β-sheet, coil) is assigned to **each residue**, not the whole
sequence — so this is a **token classification** task rather than sequence classification.
We use `AutoModelForTokenClassification`, which puts a classification head on top of *every*
token's hidden state.

The key wrinkle is **label alignment**: the tokenizer adds special tokens (`<cls>`/`<eos>`
for ESM, `[CLS]`/`[SEP]` for ProtBERT), which have no residue label. We set those positions
to `-100` so PyTorch's cross-entropy ignores them, and use `DataCollatorForTokenClassification`
to pad the label sequences alongside the inputs.

In [ ]:
# ---- Secondary Structure Prediction (Token Classification) ----
# Unlike solubility (one label per sequence), secondary structure assigns a label to
# EVERY residue. That is token classification: AutoModelForTokenClassification + a
# per-residue label sequence, with special tokens masked out using -100.
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
from sklearn.metrics import f1_score

SS_LABELS = ["H", "E", "C"]   # 3-state DSSP: Helix, sheet (Extended), Coil

def needs_spaces(name):
    return ("prot_bert" in name) or ("prot_t5" in name)

# Synthetic per-residue labels: one integer in {0,1,2} for each amino acid.
def make_ss_labels(seq):
    return [random.randint(0, len(SS_LABELS) - 1) for _ in seq]

ss_train = train_dataset.map(lambda ex: {"ss": make_ss_labels(ex["sequence"])})
ss_test  = test_dataset.map(lambda ex: {"ss": make_ss_labels(ex["sequence"])})

def tokenize_and_align(examples):
    # ESM and (space-separated) ProtBERT are char-level: exactly one token per residue,
    # flanked by a single leading special token (<cls>/[CLS]) and a trailing one (<eos>/[SEP]).
    # Special-token positions get label -100 so the loss ignores them.
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for seq, ss in zip(examples["sequence"], examples["ss"]):
        res, ss = list(seq)[:510], ss[:510]
        text = " ".join(res) if needs_spaces(MODEL_NAME) else "".join(res)
        enc = tokenizer(text, truncation=True, max_length=512)
        ids = enc["input_ids"]
        labels = [-100] * len(ids)
        for i, lab in enumerate(ss):
            pos = 1 + i                       # skip leading special token
            if pos < len(ids) - 1:            # keep trailing special token masked
                labels[pos] = lab
        out["input_ids"].append(ids)
        out["attention_mask"].append(enc["attention_mask"])
        out["labels"].append(labels)
    return out

tok_ss_train = ss_train.map(tokenize_and_align, batched=True, remove_columns=ss_train.column_names)
tok_ss_test  = ss_test.map(tokenize_and_align, batched=True, remove_columns=ss_test.column_names)

ss_model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(SS_LABELS))

def compute_ss_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=-1)
    true, pred = [], []
    for p_row, l_row in zip(preds, labels):       # flatten, dropping -100 positions
        for p, l in zip(p_row, l_row):
            if l != -100:
                true.append(l); pred.append(p)
    return {
        "accuracy": accuracy_score(true, pred),
        "f1_macro": f1_score(true, pred, average="macro", zero_division=0),
    }

ss_args = TrainingArguments(
    output_dir="./results_ss", eval_strategy="epoch", save_strategy="no",
    num_train_epochs=2, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    learning_rate=2e-5, weight_decay=0.01, logging_steps=50, report_to="none",
)
ss_trainer = Trainer(
    model=ss_model, args=ss_args,
    train_dataset=tok_ss_train, eval_dataset=tok_ss_test,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_ss_metrics,
)

print("Fine-tuning for secondary-structure (token classification)...")
ss_trainer.train()
print("\nEvaluation:", ss_trainer.evaluate())

## 8. Protein Function Prediction (Multi-label Classification)

A single protein often performs **several functions simultaneously** (e.g. multiple Gene
Ontology terms, or both "binding" and "catalysis"). This is **multi-label** classification:
each example is labelled with a *set* of classes, represented as a multi-hot vector.

We still use `AutoModelForSequenceClassification`, but with two changes:
- `problem_type="multi_label_classification"` — the model applies a **sigmoid per label**
  and optimises binary-cross-entropy, so labels are *not* mutually exclusive.
- **Labels are float multi-hot vectors** of length `num_labels`, and we threshold the
  sigmoid outputs at 0.5 at inference time. We report **micro/macro-F1** (the standard
  for multi-label) plus subset accuracy.

In [ ]:
# ---- Protein Function Prediction (Multi-label Classification) ----
# A protein can have several functions at once (e.g. multiple GO terms), so each
# example gets a multi-hot target vector and we use problem_type="multi_label_classification"
# (sigmoid + binary-cross-entropy per label, instead of a single softmax).
from sklearn.metrics import f1_score

FUNCTION_LABELS = ["binding", "catalysis", "transport", "structural", "signaling"]
n_functions = len(FUNCTION_LABELS)

def needs_spaces(name):
    return ("prot_bert" in name) or ("prot_t5" in name)

def prep_seq(seq):
    return " ".join(list(seq)) if needs_spaces(MODEL_NAME) else seq

# Synthetic multi-hot labels (floats, as BCE loss requires) in a column named "labels".
def make_function_labels(_seq):
    return [float(random.random() > 0.6) for _ in range(n_functions)]

fn_train = train_dataset.map(lambda ex: {"labels": make_function_labels(ex["sequence"])},
                             remove_columns=["label"])
fn_test  = test_dataset.map(lambda ex: {"labels": make_function_labels(ex["sequence"])},
                            remove_columns=["label"])

def tokenize_fn(examples):
    return tokenizer([prep_seq(s) for s in examples["sequence"]],
                     truncation=True, max_length=512)

tok_fn_train = fn_train.map(tokenize_fn, batched=True, remove_columns=["sequence"])
tok_fn_test  = fn_test.map(tokenize_fn, batched=True, remove_columns=["sequence"])

fn_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_functions, problem_type="multi_label_classification")

def compute_fn_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))            # sigmoid
    preds = (probs > 0.5).astype(int)
    labels = np.array(labels).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "subset_accuracy": float((preds == labels).all(axis=1).mean()),
    }

fn_args = TrainingArguments(
    output_dir="./results_function", eval_strategy="epoch", save_strategy="no",
    num_train_epochs=2, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    learning_rate=2e-5, weight_decay=0.01, logging_steps=50, report_to="none",
)
fn_trainer = Trainer(
    model=fn_model, args=fn_args,
    train_dataset=tok_fn_train, eval_dataset=tok_fn_test,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_fn_metrics,
)

print("Fine-tuning for protein function (multi-label classification)...")
fn_trainer.train()
print("\nEvaluation:", fn_trainer.evaluate())

## 9. Stability Prediction (Regression)

Stability prediction asks *how* stable a protein is — a continuous value (e.g. ΔG of
folding or a melting-temperature proxy) rather than a discrete class. This is a
**regression** task: we attach a single-output head (`num_labels=1`) and set
`problem_type="regression"`, so the `Trainer` optimises mean-squared error instead of
cross-entropy.

Key differences from classification:
- **Labels are floats**, not integers, and live in a column named `labels`.
- **Metrics** are regression metrics — MSE, MAE, and Spearman ρ (rank correlation,
  the standard for stability benchmarks like FireProt / Meltome).

In [ ]:
# ---- Stability Prediction (Regression) ----
# Predict a continuous stability value (e.g. a dG / melting-temperature proxy)
# with a single regression head: num_labels=1, problem_type="regression".
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

def needs_spaces(name):
    return ("prot_bert" in name) or ("prot_t5" in name)

def prep_seq(seq):
    return " ".join(list(seq)) if needs_spaces(MODEL_NAME) else seq

# Synthetic continuous targets (one float per sequence). Trainer's regression
# loss (MSE) requires float labels in a column literally named "labels".
st_train = train_dataset.map(lambda ex: {"labels": random.uniform(-5, 5)}, remove_columns=["label"])
st_test  = test_dataset.map(lambda ex: {"labels": random.uniform(-5, 5)}, remove_columns=["label"])

def tokenize_st(examples):
    return tokenizer([prep_seq(s) for s in examples["sequence"]],
                     truncation=True, max_length=512)

tok_st_train = st_train.map(tokenize_st, batched=True, remove_columns=["sequence"])
tok_st_test  = st_test.map(tokenize_st, batched=True, remove_columns=["sequence"])

st_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression")

def compute_st_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.array(preds).squeeze()
    labels = np.array(labels).squeeze()
    rho = spearmanr(preds, labels).correlation
    return {
        "mse": mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "spearman": float(rho) if rho == rho else 0.0,   # NaN-safe
    }

st_args = TrainingArguments(
    output_dir="./results_stability", eval_strategy="epoch", save_strategy="no",
    num_train_epochs=2, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    learning_rate=2e-5, weight_decay=0.01, logging_steps=50, report_to="none",
)
st_trainer = Trainer(
    model=st_model, args=st_args,
    train_dataset=tok_st_train, eval_dataset=tok_st_test,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_st_metrics,
)

print("Fine-tuning for stability prediction (regression)...")
st_trainer.train()
print("\nEvaluation:", st_trainer.evaluate())

## 10. Compare All 4 Models on All 4 Tasks

This final cell benchmarks **every model against every task** and prints a per-task
comparison table so you can see which pLM does best on each problem type:

| Task | Type | Reported metrics |
|------|------|------------------|
| Solubility | Binary sequence classification | accuracy, F1 |
| Secondary structure | Token classification (per-residue H/E/C) | token accuracy, macro-F1 |
| Function | Multi-label classification | micro-F1, macro-F1 |
| Stability | Regression | MSE, MAE, Spearman ρ |

Models compared: **ESM-2 (8M)**, **ESM-2 (35M)**, **ESM-2 (150M)**, **ProtBERT**.

> ⚠️ **Do not run this in the sandbox.** It fine-tunes 16 models (4 × 4) and is meant
> to be executed separately, ideally on a GPU. The cell is self-contained — it
> regenerates its own synthetic data and tokenizers, so it can run on its own.
> Each task uses the *same* synthetic labels across all models for a fair comparison.
> Because the data is random noise, expect near-chance scores; the point is to
> exercise the full pipeline for each task type. Swap in real datasets and bump
> `N_TRAIN` / `EPOCHS` to get meaningful numbers.

In [ ]:
# =====================================================================
#  COMPARISON: all 4 models  x  all 4 tasks
#  WARNING: DO NOT RUN inside the sandbox -- this fine-tunes 16 models
#  (4 models x 4 tasks). Run it separately, ideally on a GPU machine.
#  It is fully self-contained: it regenerates its own synthetic data and
#  does not depend on any variable defined in the cells above.
# =====================================================================
import time, random
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForTokenClassification, TrainingArguments, Trainer,
    DataCollatorWithPadding, DataCollatorForTokenClassification,
)
from sklearn.metrics import (
    accuracy_score, f1_score, mean_squared_error, mean_absolute_error,
)
from scipy.stats import spearmanr

# ---- Models to benchmark -------------------------------------------------
COMPARISON_MODELS = {
    "ESM-2 (8M)":   "facebook/esm2_t6_8M_UR50D",
    "ESM-2 (35M)":  "facebook/esm2_t12_35M_UR50D",
    "ESM-2 (150M)": "facebook/esm2_t30_150M_UR50D",
    "ProtBERT":     "Rostlab/prot_bert",
}

# ---- Tiny synthetic benchmark data (kept small; scale up off-sandbox) ----
N_TRAIN, N_TEST, EPOCHS = 200, 50, 1
AA = "ACDEFGHIKLMNPQRSTVWY"
FUNCTION_LABELS = ["binding", "catalysis", "transport", "structural", "signaling"]
rng = random.Random(42)

def rand_seq(n):
    return "".join(rng.choice(AA) for _ in range(n))

seqs_train = [rand_seq(rng.randint(50, 120)) for _ in range(N_TRAIN)]
seqs_test  = [rand_seq(rng.randint(50, 120)) for _ in range(N_TEST)]

# Labels are generated ONCE here so every model is scored on the same targets.
SOL_TR = [rng.randint(0, 1) for _ in seqs_train]
SOL_TE = [rng.randint(0, 1) for _ in seqs_test]
SS_TR  = [[rng.randint(0, 2) for _ in s] for s in seqs_train]   # per-residue: H/E/C
SS_TE  = [[rng.randint(0, 2) for _ in s] for s in seqs_test]
FN_TR  = [[float(rng.random() > 0.6) for _ in FUNCTION_LABELS] for _ in seqs_train]
FN_TE  = [[float(rng.random() > 0.6) for _ in FUNCTION_LABELS] for _ in seqs_test]
ST_TR  = [rng.uniform(-5, 5) for _ in seqs_train]
ST_TE  = [rng.uniform(-5, 5) for _ in seqs_test]

def needs_spaces(name):
    return ("prot_bert" in name) or ("prot_t5" in name)

def prep(seq, name):
    # ProtBERT/ProtT5 expect space-separated residues; ESM does not.
    return " ".join(seq) if needs_spaces(name) else seq

def base_args(out):
    return TrainingArguments(
        output_dir=out, num_train_epochs=EPOCHS,
        per_device_train_batch_size=8, per_device_eval_batch_size=8,
        learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="no", save_strategy="no", logging_strategy="no",
        report_to="none",
    )

# ---- Per-task runners: each fine-tunes one model and returns {metric: value} ----

def run_solubility(model_name, tok):
    def tk(b): return tok([prep(s, model_name) for s in b["sequence"]], truncation=True, max_length=512)
    dtr = Dataset.from_dict({"sequence": seqs_train, "labels": SOL_TR}).map(tk, batched=True, remove_columns=["sequence"])
    dte = Dataset.from_dict({"sequence": seqs_test,  "labels": SOL_TE}).map(tk, batched=True, remove_columns=["sequence"])
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification")
    tr = Trainer(model=model, args=base_args("./cmp_sol"), train_dataset=dtr,
                 data_collator=DataCollatorWithPadding(tok))
    tr.train()
    pred = np.argmax(tr.predict(dte).predictions, axis=1)
    return {"accuracy": accuracy_score(SOL_TE, pred),
            "f1": f1_score(SOL_TE, pred, zero_division=0)}

def run_secondary_structure(model_name, tok):
    def build(seqs, label_lists):
        rows = {"input_ids": [], "attention_mask": [], "labels": []}
        for s, ss in zip(seqs, label_lists):
            res, ss = list(s)[:510], ss[:510]
            text = " ".join(res) if needs_spaces(model_name) else "".join(res)
            enc = tok(text, truncation=True, max_length=512)
            ids = enc["input_ids"]
            lab = [-100] * len(ids)
            for i, v in enumerate(ss):                 # char-level tokenizers emit
                pos = 1 + i                            # 1 token per residue after the
                if pos < len(ids) - 1:                 # single leading special token
                    lab[pos] = v
            rows["input_ids"].append(ids)
            rows["attention_mask"].append(enc["attention_mask"])
            rows["labels"].append(lab)
        return Dataset.from_dict(rows)
    dtr, dte = build(seqs_train, SS_TR), build(seqs_test, SS_TE)
    model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=3)
    tr = Trainer(model=model, args=base_args("./cmp_ss"), train_dataset=dtr,
                 data_collator=DataCollatorForTokenClassification(tok))
    tr.train()
    out = tr.predict(dte)
    preds, labels = np.argmax(out.predictions, axis=-1), out.label_ids
    t, p = [], []
    for pr, lr in zip(preds, labels):
        for a, b in zip(pr, lr):
            if b != -100:
                t.append(b); p.append(a)
    return {"accuracy": accuracy_score(t, p),
            "f1_macro": f1_score(t, p, average="macro", zero_division=0)}

def run_function(model_name, tok):
    def tk(b): return tok([prep(s, model_name) for s in b["sequence"]], truncation=True, max_length=512)
    dtr = Dataset.from_dict({"sequence": seqs_train, "labels": FN_TR}).map(tk, batched=True, remove_columns=["sequence"])
    dte = Dataset.from_dict({"sequence": seqs_test,  "labels": FN_TE}).map(tk, batched=True, remove_columns=["sequence"])
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(FUNCTION_LABELS), problem_type="multi_label_classification")
    tr = Trainer(model=model, args=base_args("./cmp_fn"), train_dataset=dtr,
                 data_collator=DataCollatorWithPadding(tok))
    tr.train()
    logits = tr.predict(dte).predictions
    probs = 1 / (1 + np.exp(-logits))
    pred, Y = (probs > 0.5).astype(int), np.array(FN_TE).astype(int)
    return {"f1_micro": f1_score(Y, pred, average="micro", zero_division=0),
            "f1_macro": f1_score(Y, pred, average="macro", zero_division=0)}

def run_stability(model_name, tok):
    def tk(b): return tok([prep(s, model_name) for s in b["sequence"]], truncation=True, max_length=512)
    dtr = Dataset.from_dict({"sequence": seqs_train, "labels": ST_TR}).map(tk, batched=True, remove_columns=["sequence"])
    dte = Dataset.from_dict({"sequence": seqs_test,  "labels": ST_TE}).map(tk, batched=True, remove_columns=["sequence"])
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1, problem_type="regression")
    tr = Trainer(model=model, args=base_args("./cmp_st"), train_dataset=dtr,
                 data_collator=DataCollatorWithPadding(tok))
    tr.train()
    pred = tr.predict(dte).predictions.squeeze()
    rho = spearmanr(pred, ST_TE).correlation
    return {"mse": mean_squared_error(ST_TE, pred),
            "mae": mean_absolute_error(ST_TE, pred),
            "spearman": float(rho) if rho == rho else 0.0}

TASKS = {
    "Solubility (binary)":          run_solubility,
    "Secondary structure (token)":  run_secondary_structure,
    "Function (multi-label)":       run_function,
    "Stability (regression)":       run_stability,
}

# ---- Run the 4 x 4 grid --------------------------------------------------
records = []
for mlabel, mname in COMPARISON_MODELS.items():
    print(f"\n=== Model: {mlabel} ({mname}) ===")
    tok = AutoTokenizer.from_pretrained(mname, do_lower_case=False)
    for tlabel, runner in TASKS.items():
        t0 = time.time()
        try:
            metrics = runner(mname, tok)
            metrics["status"] = "ok"
        except Exception as e:
            metrics = {"status": f"error: {type(e).__name__}: {e}"}
        metrics.update({"seconds": round(time.time() - t0, 1),
                        "model": mlabel, "task": tlabel})
        records.append(metrics)
        print(f"  {tlabel:30s} -> {metrics}")

results_df = pd.DataFrame(records)

# ---- Per-task comparison tables (one row per model) ----------------------
print("\n\n================  COMPARISON BY TASK  ================")
for tlabel in TASKS:
    sub = results_df[results_df["task"] == tlabel].set_index("model")
    sub = sub.drop(columns=["task"]).dropna(axis=1, how="all")
    print(f"\n### {tlabel}")
    print(sub.to_string())

# `results_df` holds the full tidy table for any further plotting/analysis.
results_df